# Keal Voice — Single-Speaker Fine-Tuning (Google Colab)

This notebook follows the **official Qwen3-TTS single-speaker fine-tuning workflow**. It trains a dedicated speaker identity from your own recordings; it is not just zero-shot cloning.

## Before you start
- Target dataset: **20–30 minutes** of clean speech; 10 minutes is only a small experiment. These are practical targets, not official minimums.
- Split recordings into individual utterances, ideally around 5–15 seconds each. Keep each clip's transcript exact.
- Use one clean 5–15 second reference clip, and keep it identical for every training row.
- Use your own voice or a voice you have explicit permission to train.
- **GPU warning:** the upstream trainer uses BF16 and FlashAttention 2. A100 is the safest option; compatible L4 may work. A T4 is not supported by this upstream training script as-is. Free Colab GPU availability is not guaranteed.

## Google Drive dataset layout
Place your WAV clips directly inside `MyDrive/KealVoice/dataset/audio/`. The preparation cell accepts either `metadata_detailed.csv` (preferred) or `metadata.csv`, supports old `wavs/filename.wav` paths and root-level filenames, and automatically normalizes audio to mono 24 kHz. Keep the transcript text exact.


In [ ]:
# Install the upstream Qwen3-TTS training code and dependencies.
!git clone https://github.com/QwenLM/Qwen3-TTS.git /content/Qwen3-TTS
%cd /content/Qwen3-TTS
!pip -q install -e .
!pip -q install accelerate tensorboard soundfile pandas librosa
!apt-get -qq update && apt-get -qq install -y ffmpeg
%cd /content/Qwen3-TTS/finetuning
!nvidia-smi


In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, csv, subprocess, shutil, textwrap
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/KealVoice')
DATASET = BASE / 'dataset'
AUDIO_DIR = DATASET / 'audio'
WORK = Path('/content/keal_voice_finetune')
WAV_DIR = WORK / 'wav_24k_mono'
OUTPUT = BASE / 'finetuned_model'
for p in [AUDIO_DIR, WAV_DIR, OUTPUT]: p.mkdir(parents=True, exist_ok=True)

METADATA_CSV = DATASET / 'metadata.csv'
REFERENCE_FILENAME = 'reference.wav'  # Change to a filename in dataset/audio/; use a clean 5–15 sec clip.
MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-1.7B-Base'  # For a smaller experiment, use Qwen/Qwen3-TTS-12Hz-0.6B-Base.
SPEAKER_NAME = 'keal'
EPOCHS = 3
BATCH_SIZE = 1
LEARNING_RATE = '2e-5'

print('Put audio clips in:', AUDIO_DIR)
print('Create CSV:', METADATA_CSV, 'with columns filename,text')
print('Selected reference:', REFERENCE_FILENAME)


## Dataset preparation rules
1. Prefer one speaker, one microphone, one room, and consistent distance from the microphone.
2. Include normal narration, questions, statements, short and long sentences, varied phonemes, and natural pauses. Do not shout or imitate a character.
3. Avoid music, echo, clipping, background voices, aggressive denoising, compression, and reverb.
4. Each CSV row must match its audio file exactly. Do not use auto-generated transcripts without proofreading.
5. Do not put the entire 30-minute recording into one CSV row. Split it into utterances and transcribe each separately.

Example `metadata.csv`:
```csv
filename,text
clip_001.mp3,Hello everyone. Welcome back to the channel.
clip_002.wav,Today we are going to understand how the internet works.
clip_003.m4a,Let us begin with a simple question.
```


In [ ]:
import pandas as pd
import soundfile as sf
import numpy as np
import csv

# Robustly load either the detailed metadata or the simple metadata.
# Detailed metadata is preferred because it retains timestamps and source info.
def read_metadata_file(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return pd.read_csv(path, encoding="utf-8-sig", sep=None, engine="python")
    except Exception as exc:
        raise ValueError(f"Could not parse {path.name}: {exc}") from exc

detailed_path = DATASET / "metadata_detailed.csv"
simple_path = DATASET / "metadata.csv"

detailed_df = read_metadata_file(detailed_path)
simple_df = read_metadata_file(simple_path)

def usable_metadata(frame):
    if frame is None or frame.empty:
        return False
    cols = {str(c).strip().lower() for c in frame.columns}
    return "text" in cols and bool({"filename", "wav_path"} & cols)

if usable_metadata(detailed_df):
    df = detailed_df.copy()
    print("Using detailed metadata:", detailed_path)
elif usable_metadata(simple_df):
    df = simple_df.copy()
    print("Using simple metadata:", simple_path)
else:
    raise ValueError(
        "No usable metadata found. Expected metadata_detailed.csv or metadata.csv "
        "with a header containing filename (or wav_path) and text."
    )

# Normalize column names and select the path column.
df.columns = [str(c).strip().lower() for c in df.columns]
path_col = "filename" if "filename" in df.columns else "wav_path"
if "text" not in df.columns:
    raise ValueError("Metadata must contain a 'text' column.")

df = df.copy()
df["text"] = df["text"].fillna("").astype(str).str.strip()
df[path_col] = df[path_col].fillna("").astype(str).str.strip()
df = df[(df[path_col] != "") & (df["text"] != "")].reset_index(drop=True)

if df.empty:
    raise ValueError("Metadata contains no usable rows after removing blank filenames/transcripts.")

# Resolve both old paths such as wavs/keal_0001.wav and new root-level paths.
def resolve_audio_path(value):
    value = value.replace("\\", "/").strip()
    rel = Path(value)
    candidates = [AUDIO_DIR / rel, AUDIO_DIR / rel.name]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None

resolved = [resolve_audio_path(x) for x in df[path_col]]
missing = [str(df.iloc[i][path_col]) for i, p in enumerate(resolved) if p is None]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} audio files referenced by metadata were not found in {AUDIO_DIR}. "
        f"Examples: {missing[:10]}"
    )

# Reject duplicate audio references; they can silently overweight training samples.
resolved_names = [p.name for p in resolved]
duplicates = pd.Series(resolved_names)[pd.Series(resolved_names).duplicated()].tolist()
if duplicates:
    raise ValueError(f"Duplicate audio filenames in metadata: {duplicates[:10]}")

# Select a useful 5–15 second reference clip when duration metadata is available.
reference_index = 0
if "duration" in df.columns:
    durations_meta = pd.to_numeric(df["duration"], errors="coerce")
    suitable = df.index[durations_meta.between(5.0, 15.0)].tolist()
    if suitable:
        reference_index = suitable[0]

reference_source = resolved[reference_index]
print("Reference source:", reference_source.name)

def convert_to_wav(src, dst):
    cmd = [
        "ffmpeg", "-nostdin", "-y", "-i", str(src), "-vn",
        "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", str(dst)
    ]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(f"Could not convert {src.name}: {p.stderr[-1500:]}")

ref_wav = WORK / "keal_reference.wav"
convert_to_wav(reference_source, ref_wav)

raw_jsonl = WORK / "train_raw.jsonl"
rows = []
durations = []

for i, (source, (_, row)) in enumerate(zip(resolved, df.iterrows()), start=1):
    transcript = str(row["text"]).strip()
    target = WAV_DIR / f"utt_{i:04d}.wav"
    convert_to_wav(source, target)

    info = sf.info(target)
    if info.samplerate != 24000 or info.channels != 1:
        raise ValueError(
            f"Audio conversion check failed for {source.name}: "
            f"{info.samplerate} Hz, {info.channels} channel(s)."
        )
    if info.duration < 1.0:
        raise ValueError(f"{source.name} is shorter than 1 second.")
    if info.duration > 20:
        print(f"WARNING: {source.name} is {info.duration:.1f}s; consider sentence-level splitting.")

    durations.append(info.duration)
    rows.append({
        "audio": str(target),
        "text": transcript,
        "ref_audio": str(ref_wav)
    })

with raw_jsonl.open("w", encoding="utf-8") as f:
    for item in rows:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

ref_info = sf.info(ref_wav)
if not 5.0 <= ref_info.duration <= 15.0:
    print(f"WARNING: Reference clip is {ref_info.duration:.1f}s; 5–15 seconds is preferred.")

print("\n" + "=" * 48)
print("KEAL VOICE DATASET PREPARATION COMPLETE")
print("=" * 48)
print(f"Training clips: {len(rows)}")
print(f"Total speech: {sum(durations) / 60:.2f} minutes")
print(f"Median clip: {np.median(durations):.2f} seconds")
print(f"Reference: {ref_wav} ({ref_info.duration:.2f}s)")
print(f"Training manifest: {raw_jsonl}")


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Select Runtime > Change runtime type > GPU.')
gpu = torch.cuda.get_device_properties(0)
print('GPU:', gpu.name, '| VRAM:', round(gpu.total_memory/1024**3,1), 'GB')
print('BF16 supported:', torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('The official upstream fine-tuning script uses BF16. This GPU is not suitable for the unmodified trainer; use a BF16-capable GPU such as A100 or compatible L4.')

# Use PyTorch SDPA instead of requiring a separately compiled FlashAttention 2 package.
trainer_file = Path('/content/Qwen3-TTS/finetuning/sft_12hz.py')
trainer_code = trainer_file.read_text(encoding='utf-8')
trainer_code = trainer_code.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
trainer_file.write_text(trainer_code, encoding='utf-8')
print('Trainer attention backend set to PyTorch SDPA.')


In [ ]:
RAW_JSONL = str(raw_jsonl)
TRAIN_JSONL = str(WORK / 'train_with_codes.jsonl')
!python prepare_data.py --device cuda:0 --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz --input_jsonl "$RAW_JSONL" --output_jsonl "$TRAIN_JSONL"
print('Tokenized training data:', TRAIN_JSONL)


## Fine-tune
Start with 3 epochs and batch size 1. Watch GPU memory. If it runs out of memory, this upstream full-model trainer may require a larger GPU; do not assume changing the batch size alone will make a T4 work. Checkpoints are saved in Google Drive.

In [ ]:
import subprocess
import shlex
import sys
from pathlib import Path

TRAIN_DIR = Path("/content/Qwen3-TTS/finetuning")
LOG_FILE = WORK / "training.log"

cmd = [
    sys.executable, "-u", "sft_12hz.py",
    "--init_model_path", MODEL_ID,
    "--output_model_path", str(OUTPUT),
    "--train_jsonl", TRAIN_JSONL,
    "--batch_size", str(BATCH_SIZE),
    "--lr", LEARNING_RATE,
    "--num_epochs", str(EPOCHS),
    "--speaker_name", SPEAKER_NAME,
]

if not Path(TRAIN_JSONL).is_file():
    raise FileNotFoundError(f"Tokenized training manifest not found: {TRAIN_JSONL}")
if not (TRAIN_DIR / "sft_12hz.py").is_file():
    raise FileNotFoundError(f"Trainer script not found: {TRAIN_DIR / 'sft_12hz.py'}")

print("Running:", " ".join(shlex.quote(x) for x in cmd), flush=True)
print("Training log:", LOG_FILE, flush=True)

# Capture both stdout and stderr so Colab shows the real trainer exception.
result = subprocess.run(
    cmd,
    cwd=str(TRAIN_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)

LOG_FILE.write_text(result.stdout, encoding="utf-8")
print("\n========== TRAINER OUTPUT (LAST 15,000 CHARACTERS) ==========")
print(result.stdout[-15000:])

if result.returncode != 0:
    print(f"\nTRAINING FAILED — exit code {result.returncode}")
    print(f"Full log saved to: {LOG_FILE}")
    raise RuntimeError(
        "Qwen3-TTS trainer failed. Read the traceback above or inspect training.log."
    )

checkpoints = sorted(
    OUTPUT.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1
)
print("\nTraining process completed.")
print("Checkpoints:", [str(p) for p in checkpoints])


In [ ]:
import sys
import subprocess
import importlib
import torch
import soundfile as sf
from pathlib import Path

# Verify training produced a checkpoint before loading inference code.
checkpoints = sorted(
    OUTPUT.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1
)
if not checkpoints:
    raise FileNotFoundError(
        f"No checkpoint-* directory found in {OUTPUT}. "
        "Training must finish successfully before inference."
    )

# Ensure this notebook's Python kernel can see the cloned Qwen3-TTS package.
QWEN_REPO = Path("/content/Qwen3-TTS")
if not (QWEN_REPO / "qwen_tts" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"Qwen3-TTS package source not found at {QWEN_REPO}. "
        "Re-run the installation cell from the updated notebook."
    )

sys.path.insert(0, str(QWEN_REPO))
importlib.invalidate_caches()

# Install editable into the active Colab kernel without changing dependency versions.
install_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(QWEN_REPO), "--no-deps"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(install_result.stdout[-5000:])
if install_result.returncode != 0:
    raise RuntimeError("Could not install Qwen3-TTS into the active Python kernel.")

from qwen_tts import Qwen3TTSModel

BEST_CHECKPOINT = checkpoints[-1]
print("Loading checkpoint:", BEST_CHECKPOINT)

tts = Qwen3TTSModel.from_pretrained(
    str(BEST_CHECKPOINT),
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

TEST_TEXT = "Hello everyone. This is a test of my personal voice model."
wavs, sr = tts.generate_custom_voice(
    text=TEST_TEXT,
    speaker=SPEAKER_NAME,
    language="English",
)

test_path = BASE / "keal_finetuned_test.wav"
sf.write(test_path, wavs[0], sr)
print("Saved test audio:", test_path)

from IPython.display import Audio, display
display(Audio(str(test_path)))
